In [0]:
%sql

USE CATALOG `abdullah-de-airbnb`;
USE SCHEMA default;

In [0]:
%sql
SELECT * FROM airbnb_reviews_bronze
WHERE comments IS NOT NULL

In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW airbnb_cities_silver(country, city) AS
SELECT
  SPLIT(
    REPLACE(metadata.file_path, 's3://abdullah-de-airbnb-raw/', ''),
    '/'
  )[0] AS country,
  SPLIT(
    REPLACE(metadata.file_path, 's3://abdullah-de-airbnb-raw/', ''),
    '/'
  )[1] AS city
FROM airbnb_listings_bronze
QUALIFY
  ROW_NUMBER() OVER (
    PARTITION BY
      SPLIT(REPLACE(metadata.file_path, 's3://abdullah-de-airbnb-raw/', ''), '/')[0],
      SPLIT(REPLACE(metadata.file_path, 's3://abdullah-de-airbnb-raw/', ''), '/')[1]
    ORDER BY metadata.file_path
  ) = 1;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW airbnb_neighbourhoods_silver(country, city, neighbourhood_group, neighbourhood) AS
SELECT
  SPLIT(
    REPLACE(metadata.file_path, 's3://abdullah-de-airbnb-raw/', ''),
    '/'
  )[0] AS country,
  SPLIT(
    REPLACE(metadata.file_path, 's3://abdullah-de-airbnb-raw/', ''),
    '/'
  )[1] AS city,
  neighbourhood_group,
  neighbourhood
FROM airbnb_neighbourhoods_bronze
QUALIFY
  ROW_NUMBER() OVER (
    PARTITION BY
      SPLIT(REPLACE(metadata.file_path, 's3://abdullah-de-airbnb-raw/', ''), '/')[0],
      SPLIT(REPLACE(metadata.file_path, 's3://abdullah-de-airbnb-raw/', ''), '/')[1],
      neighbourhood_group,
      neighbourhood
    ORDER BY
      metadata.file_path
  ) = 1;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW airbnb_hosts_silver(
  id,
  host_url,
  host_name,
  host_since,
  host_location,
  host_about,
  host_response_time,
  host_response_rate,
  host_acceptance_rate,
  host_is_superhost,
  host_thumbnail_url,
  host_picture_url,
  host_neighbourhood,
  host_total_listings_count,
  host_verifications,
  host_has_profile_pic,
  host_identity_verified
) AS
WITH sorted_listings AS (
  SELECT * FROM airbnb_listings_bronze
  ORDER BY last_scraped
)
SELECT
  host_id,
  host_url,
  host_name,
  host_since::DATE AS host_since,
  host_location,
  TRIM(host_about) AS host_about,
  host_response_time AS host_response_time,
  CASE
  WHEN host_response_rate = 'N/A' THEN
    NULL
  ELSE
    REPLACE(host_response_rate, '%', '')::INTEGER
  END AS host_response_rate,
  CASE
  WHEN host_acceptance_rate = 'N/A' THEN
    NULL
  ELSE
    REPLACE(host_acceptance_rate, '%', '')::INTEGER
  END AS host_acceptance_rate,
  CASE
  WHEN host_is_superhost = 't' THEN
    TRUE
  ELSE
    FALSE
  END AS host_is_superhost,
  host_thumbnail_url AS host_thumbnail_url,
  host_picture_url AS host_picture_url,
  host_neighbourhood AS host_neighbourhood,
  COALESCE(host_total_listings_count, calculated_host_listings_count) AS host_total_listings_count,
  TRANSFORM(
    SPLIT(
      REGEXP_REPLACE(host_verifications, "[\\[\\]']", ""),
      ", "
    ),
    x -> TRIM(x)
  ) AS host_verifications,
  CASE
  WHEN host_has_profile_pic = 't' THEN
    TRUE
  ELSE
    FALSE
  END AS host_has_profile_pic,
  CASE
  WHEN host_identity_verified = 't' THEN
    TRUE
  ELSE
    FALSE
  END AS host_identity_verified
FROM airbnb_listings_bronze
QUALIFY
  ROW_NUMBER() OVER (
    PARTITION BY
      host_id
    ORDER BY
      last_scraped
  ) = 1;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW airbnb_listings_silver (
  id,
  listing_url,
  name,
  description,
  neighbourhood_overview,
  picture_url,
  host_id,
  country,
  city,
  neighbourhood_group,
  neighbourhood,
  coordinates,
  property_type,
  room_type,
  accommodates,
  bathrooms,
  bedrooms,
  beds,
  amenities,
  price,
  maximum_nights,
  minimum_nights,
  minimum_minimum_nights,
  maximum_minimum_nights,
  minimum_maximum_nights,
  maximum_maximum_mights,
  estimated_occupancy,
  estimated_revenue,
  review_scores_rating,
  review_scores_accuracy,
  review_scores_communication,
  review_scores_checkin,
  review_scores_cleanliness,
  review_scores_location,
  review_scores_value,
  instant_bookable,
  reviews_per_month
) AS
SELECT
  id,
  listing_url,
  name,
  description,
  neighborhood_overview,
  picture_url,
  host_id,
  SPLIT(
    REPLACE(metadata.file_path, 's3://abdullah-de-airbnb-raw/', ''),
    '/'
  )[0] AS country,
  SPLIT(
    REPLACE(metadata.file_path, 's3://abdullah-de-airbnb-raw/', ''),
    '/'
  )[1] AS city,
  neighbourhood_group_cleansed AS neighbourhood_group,
  neighbourhood_cleansed AS neighbourhood,
  STRUCT(latitude::DOUBLE, longitude::DOUBLE) AS coordinates,
  property_type,
  room_type,
  accommodates::INTEGER,
  bathrooms::DOUBLE::INTEGER,
  bedrooms::INTEGER,
  beds::INTEGER,
  TRANSFORM(
    SPLIT(
      REGEXP_REPLACE(amenities, "[\\[\\]']", ""),
      ", "
    ),
    x -> TRIM(x)
  ) AS amenities,
  REGEXP_REPLACE(price, '[^0-9.]', '')::DOUBLE,
  maximum_nights::INTEGER,
  minimum_nights::INTEGER,
  minimum_minimum_nights::INTEGER,
  maximum_minimum_nights::INTEGER,
  minimum_maximum_nights::INTEGER,
  maximum_maximum_nights::INTEGER,
  estimated_occupancy_l365d::DOUBLE AS estimated_occupancy,
  estimated_revenue_l365d::DOUBLE AS estimated_revenue,
  review_scores_rating::DOUBLE,
  review_scores_accuracy::DOUBLE,
  review_scores_communication::DOUBLE,
  review_scores_checkin::DOUBLE,
  review_scores_cleanliness::DOUBLE,
  review_scores_location::DOUBLE,
  review_scores_value::DOUBLE,
  CASE
  WHEN instant_bookable = 't' THEN
    TRUE
  ELSE
    FALSE
  END AS instant_bookable,
  reviews_per_month::DOUBLE
FROM
  airbnb_listings_bronze
QUALIFY
  ROW_NUMBER() OVER (
    PARTITION BY
      id
    ORDER BY last_scraped
  ) = 1;


  

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW airbnb_reviewers_silver (
  id,
  reviewer_name
) AS
SELECT 
  reviewer_id,
  reviewer_name
FROM airbnb_reviews_bronze
WHERE reviewer_id IS NOT NULL
QUALIFY
  ROW_NUMBER() OVER (
    PARTITION BY
      reviewer_id
    ORDER BY date
  ) = 1;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW airbnb_reviews_silver (
  listing_id,
  id,
  date,
  reviewer_id,
  comments
) AS
SELECT
  listing_id,
  id,
  date::DATE,
  reviewer_id,
  comments
FROM airbnb_reviews_bronze
QUALIFY
  ROW_NUMBER() OVER (
    PARTITION BY
      listing_id,
      id
    ORDER BY date
  ) = 1;